# 04. Model Training & Evaluation

This notebook:
1. Loads preprocessed data from Notebook 03
2. Trains multiple classifiers (Random Forest, XGBoost, Gradient Boosting)
3. Evaluates and compares models
4. Saves the best model and all artifacts

In [3]:
import pandas as pd
import numpy as np
import joblib
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix
)

try:
    import xgboost as xgb
    HAS_XGBOOST = True
except ImportError:
    HAS_XGBOOST = False
    print("XGBoost not installed, skipping XGBoost models")

XGBoostError: 
XGBoost Library (libxgboost.dylib) could not be loaded.
Likely causes:
  * OpenMP runtime is not installed
    - vcomp140.dll or libgomp-1.dll for Windows
    - libomp.dylib for Mac OSX
    - libgomp.so for Linux and other UNIX-like OSes
    Mac OSX users: Run `brew install libomp` to install OpenMP runtime.

  * You are running 32-bit Python on a 64-bit OS

Error message(s): ["dlopen(/Users/apple/Desktop/AcadAlert/venv/lib/python3.13/site-packages/xgboost/lib/libxgboost.dylib, 0x0006): Library not loaded: @rpath/libomp.dylib\n  Referenced from: <5A4ADBF3-2060-3ECB-AB2B-F8A248DDA0A7> /Users/apple/Desktop/AcadAlert/venv/lib/python3.13/site-packages/xgboost/lib/libxgboost.dylib\n  Reason: tried: '/usr/local/opt/libomp/lib/libomp.dylib' (no such file), '/System/Volumes/Preboot/Cryptexes/OS/usr/local/opt/libomp/lib/libomp.dylib' (no such file), '/usr/local/opt/libomp/lib/libomp.dylib' (no such file), '/System/Volumes/Preboot/Cryptexes/OS/usr/local/opt/libomp/lib/libomp.dylib

## 1. Load Preprocessed Data

In [ ]:
# Load preprocessed train/test data
train_data = joblib.load('../data/processed/train_test_data.joblib')
X_train = train_data['X_train']
X_test = train_data['X_test']
y_train = train_data['y_train']
y_test = train_data['y_test']

# Load feature config
feature_config = joblib.load('../models/feature_config.joblib')
label_encoder = joblib.load('../models/label_encoder.joblib')

print(f"Training set: {X_train.shape[0]} samples, {X_train.shape[1]} features")
print(f"Test set: {X_test.shape[0]} samples, {X_test.shape[1]} features")
print(f"\nFeatures: {feature_config['all_features']}")
print(f"Classes: {list(label_encoder.classes_)}")

## 2. Define Models

In [ ]:
# Define models to compare
models = {
    'Logistic Regression': LogisticRegression(
        max_iter=1000, random_state=42, multi_class='multinomial'
    ),
    'Random Forest': RandomForestClassifier(
        n_estimators=100, max_depth=10, random_state=42
    ),
    'Gradient Boosting': GradientBoostingClassifier(
        n_estimators=100, max_depth=5, random_state=42
    )
}

if HAS_XGBOOST:
    models['XGBoost'] = xgb.XGBClassifier(
        n_estimators=100, max_depth=5, learning_rate=0.1,
        random_state=42, use_label_encoder=False, eval_metric='mlogloss'
    )

print("Models defined:")
for name in models.keys():
    print(f"  - {name}")

## 3. Train and Evaluate

In [ ]:
# Train and evaluate each model
results = []

for name, model in models.items():
    print(f"\n{'='*50}")
    print(f"Training: {name}")
    print('='*50)
    
    # Train
    model.fit(X_train, y_train)
    
    # Predictions
    y_pred = model.predict(X_test)
    y_pred_proba = model.predict_proba(X_test)
    
    # Metrics
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, average='weighted')
    recall = recall_score(y_test, y_pred, average='weighted')
    f1 = f1_score(y_test, y_pred, average='weighted')
    
    # Cross-validation
    cv_scores = cross_val_score(model, X_train, y_train, cv=5, scoring='accuracy')
    
    results.append({
        'model_name': name,
        'model': model,
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1_score': f1,
        'cv_mean': cv_scores.mean(),
        'cv_std': cv_scores.std()
    })
    
    print(f"Accuracy:  {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall:    {recall:.4f}")
    print(f"F1 Score:  {f1:.4f}")
    print(f"CV Score:  {cv_scores.mean():.4f} (+/- {cv_scores.std()*2:.4f})")

## 4. Compare Models

In [ ]:
# Create results dataframe
results_df = pd.DataFrame(results)
results_df = results_df.sort_values('f1_score', ascending=False)

print("\nModel Comparison (sorted by F1 Score):")
display(results_df[['model_name', 'accuracy', 'precision', 'recall', 'f1_score', 'cv_mean']])

In [ ]:
# Visualize comparison
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(12, 6))
metrics = ['accuracy', 'precision', 'recall', 'f1_score']
x = np.arange(len(results_df))
width = 0.2

for i, metric in enumerate(metrics):
    plt.bar(x + i*width, results_df[metric], width, label=metric.replace('_', ' ').title())

plt.xlabel('Model')
plt.ylabel('Score')
plt.title('Model Performance Comparison', fontsize=14, fontweight='bold')
plt.xticks(x + width*1.5, results_df['model_name'], rotation=45)
plt.legend()
plt.ylim(0, 1)
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('../models/model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Best Model Evaluation

In [ ]:
# Select best model
best_model_info = results_df.iloc[0]
best_model = best_model_info['model']
best_model_name = best_model_info['model_name']

print(f"Best Model: {best_model_name}")
print(f"F1 Score: {best_model_info['f1_score']:.4f}")

In [ ]:
# Detailed evaluation of best model
y_pred_best = best_model.predict(X_test)

print(f"\nClassification Report - {best_model_name}:")
print(classification_report(y_test, y_pred_best, target_names=label_encoder.classes_))

print("\nConfusion Matrix:")
cm = confusion_matrix(y_test, y_pred_best)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=label_encoder.classes_,
            yticklabels=label_encoder.classes_)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title(f'Confusion Matrix - {best_model_name}')
plt.tight_layout()
plt.savefig('../models/confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Save Model Artifacts

In [ ]:
# Save the best model
joblib.dump(best_model, '../models/classifier.joblib')
print(f"Saved model: ../models/classifier.joblib")

# Update feature info with model name
feature_config['model_name'] = best_model_name
feature_config['model_type'] = type(best_model).__name__
joblib.dump(feature_config, '../models/feature_config.joblib')
print(f"Updated: ../models/feature_config.joblib")

# Save test set predictions for reference
test_results = pd.DataFrame({
        'y_true': y_test,
        'y_pred': y_pred_best,
        'y_pred_proba': best_model.predict_proba(X_test).tolist()
    })
joblib.dump(test_results, '../data/processed/test_predictions.joblib')
print(f"Saved predictions: ../data/processed/test_predictions.joblib")

In [ ]:
# Verify all saved files
import os

print("\n" + "="*60)
print("MODEL TRAINING COMPLETE")
print("="*60)
print(f"\nBest Model: {best_model_name}")
print(f"F1 Score: {best_model_info['f1_score']:.4f}")
print(f"Accuracy: {best_model_info['accuracy']:.4f}")

print("\nSaved files in ../models/:")
for f in sorted(os.listdir('../models/')):
    size = os.path.getsize(f'../models/{f}')
    print(f"  {f}: {size:,} bytes")

print("\nNext: Run 05_model_explainability.ipynb for SHAP analysis")